# SAD AR5 — Notebook completo do pipeline analítico

**CT302 · Grupo VI · Escola Naval**

Este notebook documenta **todo o código** usado para obter os valores, figuras e JSON que alimentam o **relatório** e a **plataforma web**.

Cada secção mostra:
1. O **ficheiro Python** correspondente (`src/…`)
2. A **execução passo a passo** com resultados intermédios
3. As **figuras** geradas (`resultados/figuras/`)

| Secção | Módulo | Saídas |
|--------|--------|--------|
| 0 | Setup | caminhos, imports |
| 1 | `dm/preproc.py` + `dm/eda.py` | EDA, `resultados/dm/eda_panorama.png` |
| 2 | `dm/ahp_pesos.py` | `ahp_pesos.json`, `24_ahp_pesos.png` |
| 3 | `geo.py` + `risco.py` | grelha, índice multi-ameaça, figs 01–02 |
| 4 | `otimizacao.py` + `main.py` | MCLP, frota, figs 03–08, `resultados.json` |
| 5 | `validacao.py` | backtest, baseline, figs 21–22–25, `validacao.json` |
| 6 | Respostas SAD | Q1, Q2, Q3 |
| 7 | Ligação à plataforma | JSON consumidos pela API |

> **Executar:** `jupyter notebook` na pasta `notebooks/` ou *Run All* no Colab após upload do repositório.


## 0. Setup

In [ ]:
# %pip install -q -r ../requirements.txt jupyter matplotlib seaborn openpyxl pulp scikit-learn shapely

import csv
import json
import math
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import seaborn as sns
from IPython.display import Image, Markdown, display, Code

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 6)
sns.set_theme(style="whitegrid")

REPO = Path.cwd()
if not (REPO / "src" / "config.py").exists():
    REPO = REPO.parent
SRC = REPO / "src"
sys.path.insert(0, str(SRC))
os.chdir(SRC)

FIGDIR = REPO / "resultados" / "figuras"
DMFIG = REPO / "resultados" / "dm"
OUTDIR = REPO / "resultados"
DADOS = REPO / "dados"
XLSX = DADOS / "fontes/apreensoes_droga_PT.xlsx"
CSV_INT = DADOS / "processados/intensidades_reais.csv"
FIGDIR.mkdir(parents=True, exist_ok=True)
DMFIG.mkdir(parents=True, exist_ok=True)

def show_fig(name, title=None, folder=FIGDIR):
    p = folder / name
    if p.exists():
        if title:
            display(Markdown(f"*{title}*"))
        display(Image(filename=str(p), width=900))
    else:
        print(f"Figura em falta: {p}")

print("Repo:", REPO)
print("Módulos src:", sorted(p.name for p in SRC.glob("*.py")))


## 1. Pré-processamento e EDA das apreensões

**Fonte:** `dados/fontes/apreensoes_droga_PT.xlsx` (formato UNODC, 2011–2024)

Pipeline CRISP-DM:
- Limpeza de inconsistências
- Geocodificação por distrito
- Agrupamento de substâncias
- Definição do alvo `maritimo` (apreensão em águas territoriais / embarcação)


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Pré-processamento** (`src/dm/preproc.py`)"))
display(Code(filename=str(REPO / "src/dm/preproc.py"), language="python"))


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Análise exploratória** (`src/dm/eda.py`)"))
display(Code(filename=str(REPO / "src/dm/eda.py"), language="python"))


In [ ]:
from dm.preproc import carregar_e_limpar, features_classificacao, GRUPO_DROGA, LOC_MARITIMO
from dm.eda import gerar as eda_gerar

df = carregar_e_limpar(str(XLSX), log=True)
X, y, d = features_classificacao(df)

print("=== Resumo do dataset limpo ===")
print(f"Registos: {len(df)}")
print(f"Anos: {int(df['ano'].min())}–{int(df['ano'].max())}")
print(f"Apreensões marítimas: {df['maritimo'].sum()} ({100*df['maritimo'].mean():.2f}%)")
print(f"Geocodificadas: {df['lat'].notna().sum()} ({100*df['lat'].notna().mean():.1f}%)")

display(df[["ano","grupo_droga","regiao","maritimo","qty_g","log_qty_g"]].describe())

# Série temporal e distribuições (mesma lógica que eda.py)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
serie = df.groupby("ano").size()
axes[0,0].plot(serie.index, serie.values, "-o"); axes[0,0].set_title("(a) Apreensões por ano")
g = df["grupo_droga"].value_counts()
sns.barplot(x=g.values, y=g.index, ax=axes[0,1], hue=g.index, legend=False)
axes[0,1].set_title("(b) Por grupo de droga")
axes[1,0].hist(df["log_qty_g"].dropna(), bins=40, color="#2ca02c", alpha=0.8)
axes[1,0].set_title("(c) log(1+quantidade g)")
mar_pct = df.groupby("grupo_droga")["maritimo"].mean().sort_values(ascending=False)*100
sns.barplot(x=mar_pct.values, y=mar_pct.index, ax=axes[1,1], hue=mar_pct.index, legend=False)
axes[1,1].set_title("(d) % marítimas por grupo")
fig.suptitle("EDA — apreensões Portugal 2011–2024", fontsize=14)
fig.tight_layout(); fig.savefig(DMFIG/"eda_panorama.png", dpi=140); plt.show()

resumo_eda = eda_gerar()
print(json.dumps(resumo_eda, indent=2, ensure_ascii=False))


## 2. Pesos AHP das ameaças

Método de Saaty (escala 1–9). A matriz compara droga, pesca INN, poluição e imigração.

**Fórmula:** autovetor principal normalizado → pesos \(w_i\); \(CR = CI/RI\) com \(CI = (\lambda_{max}-n)/(n-1)\).


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — AHP — pesos multicritério** (`src/dm/ahp_pesos.py`)"))
display(Code(filename=str(REPO / "src/dm/ahp_pesos.py"), language="python"))


In [ ]:
from dm.ahp_pesos import MATRIZ, NOMES, LABELS, _cr, fig_ahp, sensibilidade_frota
from config import PESOS_AMEACA

cr, vec = _cr(MATRIZ)
pesos_ahp = {n: round(float(v), 4) for n, v in zip(NOMES, vec)}
s = sum(pesos_ahp.values())
pesos_ahp = {k: round(v/s, 4) for k, v in pesos_ahp.items()}

print("Matriz de comparação par a par (Saaty):")
display(pd.DataFrame(MATRIZ, index=LABELS, columns=LABELS))

print(f"\nCR = {cr:.4f}  (< 0.10 → consistente)")
display(pd.DataFrame({
    "Ameaça": NOMES,
    "Peso AHP": [pesos_ahp[k] for k in NOMES],
    "Peso adoptado (config)": [PESOS_AMEACA[k] for k in NOMES],
}))

fig_ahp(pesos_ahp, cr)
show_fig("24_ahp_pesos.png", "Fig. 24 — Pesos AHP")

sens = sensibilidade_frota(PESOS_AMEACA)
print("Sensibilidade ±10% no nº células alto risco:", json.dumps(sens, indent=2))


## 3. Grelha marítima e índice de risco

### 3.1 Grelha de procura (`geo.py`)
Células de 0,10° em faixa costeira 8–300 km, máscara de terra Shapely.

### 3.2 Índice multi-ameaça (`risco.py`)

$$R_i = \frac{\sum_k w_k \cdot I_{k,i}}{\max_j \sum_k w_k \cdot I_{k,j}}$$

onde \(I_k\) são intensidades normalizadas [0,1] de droga, pesca, poluição e imigração.
Fonte preferencial: `dados/processados/intensidades_reais.csv` (EMODnet + IOM + apreensões).


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Geografia e grelha** (`src/geo.py`)"))
display(Code(filename=str(REPO / "src/geo.py"), language="python"))


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Índice de risco multi-ameaça** (`src/risco.py`)"))
display(Code(filename=str(REPO / "src/risco.py"), language="python"))


In [ ]:
from geo import gerar_procura, bases_proj, COSTA_LONLAT, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX, proj, _KX, _KY
from risco import calcular_risco, _carregar_intensidades_reais, aplicar_pesos
from config import PESOS_AMEACA, LIMIAR_RISCO_OPERACIONAL, AERODROMOS

LIMIAR = LIMIAR_RISCO_OPERACIONAL
pts = gerar_procura()
bases = bases_proj()
print(f"Grelha: {len(pts)} células marítimas | {len(bases)} bases candidatas")

# Verificar intensidades reais
reais = _carregar_intensidades_reais(pts)
print("Intensidades reais (EMODnet/IOM):", "SIM" if reais else "fallback gaussiano")

calcular_risco(pts, str(XLSX))
r = np.array([p["risco"] for p in pts])

stats = pd.Series({
    "células": len(pts),
    "risco médio": round(r.mean(), 4),
    "risco máx.": round(r.max(), 4),
    f"alto risco (≥{LIMIAR})": int((r >= LIMIAR).sum()),
    "% alto risco": round(100*(r>=LIMIAR).mean(), 1),
}, name="Grelha SAD")
display(stats)

rows = []
for a in ["droga","pesca","poluicao","imigracao"]:
    v = np.array([p[f"r_{a}"] for p in pts])
    rows.append({"ameaça": a, "peso": PESOS_AMEACA[a],
                 "média": round(v.mean(),3), "máx": round(v.max(),3),
                 "p90": round(np.percentile(v,90),3)})
display(pd.DataFrame(rows))


In [ ]:
# --- Fig. 01: mapa de risco agregado (código de main.py/fig_risco) ---
COSTA_LON = [c[0] for c in COSTA_LONLAT]; COSTA_LAT = [c[1] for c in COSTA_LONLAT]
lon = [p["lon"] for p in pts]; lat = [p["lat"] for p in pts]; rv = [p["risco"] for p in pts]

fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(lon, lat, c=rv, cmap="YlOrRd", s=14, marker="s", vmin=0, vmax=1)
for nome, blon, blat, reg in AERODROMOS:
    ax.plot(blon, blat, "^", color="navy", ms=8)
ax.plot(COSTA_LON, COSTA_LAT, color="#444", lw=1.5)
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_aspect(1.0/np.cos(np.radians(39.5)))
plt.colorbar(sc, ax=ax, label="Risco [0–1]", shrink=0.7)
ax.set_title("Fig. 01 — Índice de risco marítimo multi-ameaça")
fig.tight_layout(); fig.savefig(FIGDIR/"01_risco.png", dpi=140); plt.show()

# --- Fig. 02: campos por ameaça ---
campos = [("r_droga","Tráfico de droga","Reds"),("r_pesca","Pesca INN","Greens"),
          ("r_poluicao","Poluição","Purples"),("r_imigracao","Imigração","Blues")]
fig, axes = plt.subplots(2, 2, figsize=(11, 12))
for (ch, tit, cmap), ax in zip(campos, axes.ravel()):
    v = [p[ch] for p in pts]
    sc = ax.scatter(lon, lat, c=v, cmap=cmap, s=8, marker="s", vmin=0, vmax=1)
    ax.plot(COSTA_LON, COSTA_LAT, color="#444", lw=1)
    ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect(1.0/np.cos(np.radians(39.5)))
    ax.set_title(f"{tit} (peso {PESOS_AMEACA[ch.split('_')[1]]:.2f})")
    plt.colorbar(sc, ax=ax, shrink=0.7)
fig.suptitle("Fig. 02 — Campos de intensidade por ameaça", fontsize=14)
fig.tight_layout(); fig.savefig(FIGDIR/"02_ameacas.png", dpi=140); plt.show()


## 4. Otimização — set cover, MCLP e dimensionamento de frota

### 4.1 Set Cover (PuLP)
Minimizar nº de bases \(y_b\) sujeito a \(\sum_b a_{bi} y_b \geq 1\) para cada célula \(i\) de alto risco cobrível.

### 4.2 MCLP
Maximizar \(\sum_i r_i z_i\) com \(z_i \leq \sum_b a_{bi} y_b\) e \(\sum_b y_b \leq k\).

### 4.3 Frota 24 h (persistência sensorial)
\(n_{sim} = \lceil A_{alto} / (V \cdot swath \cdot T_{rev}) \rceil\), rotação \(M = 24 / (t_{on} \cdot disp)\).


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Otimização PuLP** (`src/otimizacao.py`)"))
display(Code(filename=str(REPO / "src/otimizacao.py"), language="python"))


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Parâmetros AR5 e pesos** (`src/config.py`)"))
display(Code(filename=str(REPO / "src/config.py"), language="python"))


In [ ]:
from otimizacao import (matriz_cobertura, set_cover, mclp, curva_tradeoff,
    dimensionar_frota, dimensionar_persistencia, raio_efetivo, raio_por_autonomia)
from config import AR5, CENARIOS_VENTO, fator_vento, SENSOR_SWATH_KM, TEMPO_REVISITA_H, T_ON_MIN_H

risco_total = sum(p["risco"] for p in pts)
area_celula = (0.10 * _KX) * (0.10 * _KY)
pts_alto = [p for p in pts if p["risco"] >= LIMIAR]
print(f"Células alto risco: {len(pts_alto)} | Área célula: {area_celula:.1f} km²")

# --- Cenário A: vento calmo/moderado/forte, R base 90 km ---
cenarios_vento = {}
for nome, vel in CENARIOS_VENTO.items():
    R = raio_efetivo(vel)
    sc = set_cover(pts, bases, R, LIMIAR)
    fr = dimensionar_frota(sc["n_bases"], R)
    cenarios_vento[f"{nome} ({vel:.0f} m/s)"] = {
        "vento_ms": vel, "raio_km": R, "set_cover": sc, "frota": fr,
        "bases_nomes": [bases[b]["nome"] for b in sc["bases_sel"]],
    }
    print(f"  {nome:8s} R={R:5.0f} km  bases={sc['n_bases']:2d}  frota={fr['frota_total']:3d}  fora_alcance={sc['n_nao_cobrivel']}")

display(pd.DataFrame([{
    "cenário": k, "R km": v["raio_km"], "bases": v["set_cover"]["n_bases"],
    "frota 24h": v["frota"]["frota_total"], "fora alcance": v["set_cover"]["n_nao_cobrivel"],
} for k,v in cenarios_vento.items()]))


In [ ]:
# --- Cenário B: MCLP + persistência ---
R_calmo = raio_efetivo(CENARIOS_VENTO["calmo"])
t_on_alvo = 6.0
R_long = raio_por_autonomia(t_on_alvo, CENARIOS_VENTO["calmo"])

c90 = curva_tradeoff(pts, bases, R_calmo)
cL = curva_tradeoff(pts, bases, R_long)

# Minimizar frota entre configs com ≥95% risco coberto
frota_vs_k = []
for c in cL:
    sel = c["bases_sel"]
    fp = dimensionar_persistencia(pts_alto, bases, sel, area_celula)
    frota_vs_k.append({"k": c["k"], "frac_risco": c["frac_risco"],
                       "frota_total": fp["frota_total"], "bases": [bases[b]["nome"] for b in sel]})

elegiveis = [f for f in frota_vs_k if f["frac_risco"] >= 0.95]
melhor = min(elegiveis, key=lambda f: f["frota_total"]) if elegiveis else frota_vs_k[-1]
k_rec = melhor["k"]
rec = mclp(pts, bases, R_long, k_rec)
bases_rec = rec["bases_sel"]

fr_persist = dimensionar_persistencia(pts_alto, bases, bases_rec, area_celula)
sc90 = set_cover(pts, bases, R_calmo, LIMIAR)
pts_alto_cost = [pts_alto[i] for i in range(len(pts_alto))
                 if matriz_cobertura(pts_alto, bases, R_calmo)[:, i].sum() > 0]
fr_persist_cost = dimensionar_persistencia(pts_alto_cost, bases, sc90["bases_sel"], area_celula)

print(f"MCLP k={k_rec}: cobre {100*rec['frac_risco']:.0f}% risco")
print(f"Bases: {', '.join(bases[b]['nome'] for b in bases_rec)}")
print(f"Frota costeira 24h: {fr_persist_cost['frota_total']} AR5 ({fr_persist_cost['n_simultaneos']} simultâneos)")
print(f"Frota total 24h: {fr_persist['frota_total']} AR5")

display(pd.DataFrame(frota_vs_k))


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Orquestração e figuras 03–08** (`src/main.py`)"))
display(Code(filename=str(REPO / "src/main.py"), language="python"))


In [ ]:
# --- Sensibilidade frota (swath, revisita, disponibilidade) ---
sens = {"Largura sensor (km)": [], "Tempo revisita (h)": [], "Disponibilidade": []}
for w in [20, 30, 40, 50]:
    d = dimensionar_persistencia(pts_alto, bases, bases_rec, area_celula, swath_km=w)
    sens["Largura sensor (km)"].append({"valor": w, "frota_total": d["frota_total"]})
for t in [2, 3, 4, 6]:
    d = dimensionar_persistencia(pts_alto, bases, bases_rec, area_celula, revisita_h=t)
    sens["Tempo revisita (h)"].append({"valor": t, "frota_total": d["frota_total"]})
for a in [0.6, 0.7, 0.8, 0.9]:
    d = dimensionar_persistencia(pts_alto, bases, bases_rec, area_celula, disponibilidade=a)
    sens["Disponibilidade"].append({"valor": a, "frota_total": d["frota_total"]})

# Exportar resultados.json (mesma estrutura que main.py)
resultados = {
    "meta": {"n_celulas": len(pts), "limiar_alto_risco": LIMIAR,
             "risco_total": risco_total, "AR5": AR5, "pesos": PESOS_AMEACA},
    "cenarios": {
        "A_conservador_vento": cenarios_vento,
        "B_alcance_AR5": {
            "R_long_km": R_long, "k_recomendado": k_rec,
            "frac_risco_recomendado": rec["frac_risco"],
            "bases_recomendadas": [bases[b]["nome"] for b in bases_rec],
            "frota_persistencia_total": fr_persist,
            "frota_persistencia_costeira": fr_persist_cost,
        },
    },
    "frota_vs_k": frota_vs_k,
    "tradeoff": {"R90": c90, "R_long": cL, "R90_km": R_calmo, "R_long_km": R_long},
    "sensibilidade": sens,
}
with open(OUTDIR/"resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2, default=float)
print("Guardado:", OUTDIR/"resultados.json")


In [ ]:
# --- Gerar figuras 03–08 (código inline de main.py) ---

def circulo_lonlat(base, raio_km, n=120):
    from geo import inv_proj
    ang = np.linspace(0, 2*np.pi, n)
    xs = base["x"] + raio_km * np.cos(ang)
    ys = base["y"] + raio_km * np.sin(ang)
    return ([inv_proj(x,y)[0] for x,y in zip(xs,ys)],
            [inv_proj(x,y)[1] for x,y in zip(xs,ys)])

sc_calmo = cenarios_vento[f"calmo ({CENARIOS_VENTO['calmo']:.0f} m/s)"]["set_cover"]

# Fig 03 cobertura conservador
a = matriz_cobertura(pts, bases, R_calmo)
sel = sc_calmo["bases_sel"]; cob = a[sel].sum(axis=0) > 0
lon_a = np.array([p["lon"] for p in pts]); lat_a = np.array([p["lat"] for p in pts])
r_a = np.array([p["risco"] for p in pts]); alto = r_a >= LIMIAR
fig, ax = plt.subplots(figsize=(7.5, 8.5))
ax.scatter(lon_a[~alto], lat_a[~alto], c="#ddd", s=6, marker="s")
ax.scatter(lon_a[alto&cob], lat_a[alto&cob], c="#1a9850", s=12, marker="s", label="Coberto")
ax.scatter(lon_a[alto&~cob], lat_a[alto&~cob], c="#d73027", s=12, marker="s", label="Não coberto")
for b in sel:
    clon, clat = circulo_lonlat(bases[b], R_calmo)
    ax.plot(clon, clat, color="navy", lw=1.2, alpha=0.8)
    ax.plot(bases[b]["lon"], bases[b]["lat"], "^", color="navy", ms=10)
ax.plot(COSTA_LON, COSTA_LAT, color="#444", lw=1.5)
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_title(f"Fig. 03 — Cobertura conservadora R={R_calmo:.0f} km")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR/"03_cobertura_conservador.png", dpi=140); plt.show()


# Fig 04 cobertura MCLP recomendado
aL = matriz_cobertura(pts, bases, R_long)
cobL = aL[bases_rec].sum(axis=0) > 0 if bases_rec else np.zeros(len(pts), bool)
fig, ax = plt.subplots(figsize=(7.5, 8.5))
ax.scatter(lon_a[~alto], lat_a[~alto], c="#ddd", s=6, marker="s")
ax.scatter(lon_a[alto&cobL], lat_a[alto&cobL], c="#1a9850", s=12, marker="s", label="Coberto")
ax.scatter(lon_a[alto&~cobL], lat_a[alto&~cobL], c="#d73027", s=12, marker="s", label="Não coberto")
for b in bases_rec:
    clon, clat = circulo_lonlat(bases[b], R_long)
    ax.plot(clon, clat, color="navy", lw=1.2, alpha=0.8)
    ax.plot(bases[b]["lon"], bases[b]["lat"], "^", color="navy", ms=10)
ax.plot(COSTA_LON, COSTA_LAT, color="#444", lw=1.5)
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_title(f"Fig. 04 — MCLP k={k_rec}, R={R_long:.0f} km")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR/"04_cobertura_alargado.png", dpi=140); plt.show()

# Fig 05 tradeoff
fig, ax = plt.subplots(figsize=(8, 5.5))
for curva, nome, R in [(c90, f"R={R_calmo:.0f} km", R_calmo), (cL, f"R={R_long:.0f} km", R_long)]:
    ax.plot([c["k"] for c in curva], [100*c["frac_risco"] for c in curva], "-o", label=nome)
ax.axhline(95, color="gray", ls="--"); ax.set_xlabel("Nº bases"); ax.set_ylabel("% risco coberto")
ax.set_title("Fig. 05 — Trade-off MCLP"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGDIR/"05_tradeoff.png", dpi=140); plt.show()

# Fig 06 frota por vento
nomes = list(cenarios_vento.keys())
nb = [cenarios_vento[n]["set_cover"]["n_bases"] for n in nomes]
ft = [cenarios_vento[n]["frota"]["frota_total"] for n in nomes]
x = np.arange(len(nomes)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x-w/2, nb, w, label="Bases simultâneas", color="#4575b4")
ax.bar(x+w/2, ft, w, label="Frota 24h", color="#d73027")
ax.set_xticks(x); ax.set_xticklabels(nomes, rotation=15, ha="right")
ax.set_title("Fig. 06 — Frota por cenário de vento"); ax.legend()
fig.tight_layout(); fig.savefig(FIGDIR/"06_frota.png", dpi=140); plt.show()

# Fig 07 sensibilidade
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (titulo, dados) in zip(axes, sens.items()):
    xs = [d["valor"] for d in dados]; ys = [d["frota_total"] for d in dados]
    ax.plot(xs, ys, "-o", color="#d73027")
    for x,y in zip(xs,ys): ax.text(x, y+0.2, str(y), ha="center", fontsize=9)
    ax.set_title(titulo); ax.set_ylabel("Frota total"); ax.grid(alpha=0.3)
fig.suptitle("Fig. 07 — Sensibilidade da frota"); fig.tight_layout()
fig.savefig(FIGDIR/"07_sensibilidade.png", dpi=140); plt.show()

# Fig 08 frota vs k
ks = [f["k"] for f in frota_vs_k]; ft2 = [f["frota_total"] for f in frota_vs_k]
fr2 = [100*f["frac_risco"] for f in frota_vs_k]
fig, ax1 = plt.subplots(figsize=(8.5, 5.5))
ax1.plot(ks, ft2, "-o", color="#d73027", label="Frota 24h")
ax1.axvline(k_rec, color="green", ls="--", label=f"Ótimo k={k_rec}")
ax2 = ax1.twinx(); ax2.plot(ks, fr2, "-s", color="#4575b4", alpha=0.7)
ax1.set_xlabel("Nº bases"); ax1.set_ylabel("Frota AR5", color="#d73027")
ax2.set_ylabel("% risco", color="#4575b4"); ax1.set_title("Fig. 08 — Frota vs nº bases")
ax1.legend(); fig.tight_layout(); fig.savefig(FIGDIR/"08_frota_vs_bases.png", dpi=140); plt.show()

# CSV resumo
with open(OUTDIR/"resumo_cenarios.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["Cenario","Vento_ms","Raio_km","N_bases","Frota_total","Bases"])
    for nome, c in cenarios_vento.items():
        w.writerow([nome, c["vento_ms"], f"{c['raio_km']:.0f}",
                    c["set_cover"]["n_bases"], c["frota"]["frota_total"],
                    "; ".join(c["bases_nomes"])])
print("Guardado:", OUTDIR/"resumo_cenarios.csv")


## 5. Validação científica

### 5.1 Backtest temporal
Treino: apreensões marítimas ≤ 2022 → campo de risco.  
Teste: holdout 2023–2024 → taxa de acerto em células de alto risco.

### 5.2 Baseline de patrulha
Compara captura de risco: **SAD top-N** vs **aleatório** vs **uniforme costeiro**.


In [ ]:
from IPython.display import display, Code
display(Markdown("**Código-fonte — Backtest e baseline** (`src/validacao.py`)"))
display(Code(filename=str(REPO / "src/validacao.py"), language="python"))


In [ ]:
from validacao import (backtest_temporal, baseline_patrulha, sensibilidade_limiar,
    fig_backtest, fig_baseline, fig_sensibilidade_limiar, caixa_objetivo, ANO_CORTE)

# Backtest (código em validacao.py — executado aqui)
bt = backtest_temporal(pts)
print("=== Backtest temporal ===")
for k,v in bt.items():
    print(f"  {k}: {v}")

display(pd.DataFrame([
    {"métrica": "Ano corte", "valor": bt.get("ano_corte")},
    {"métrica": "N holdout", "valor": bt.get("n_holdout")},
    {"métrica": "Acerto limiar", "valor": f"{bt.get('taxa_acerto_limiar',0):.1%}"},
    {"métrica": "Ganho vs aleatório", "valor": f"{bt.get('ganho_relativo_limiar',0)}×"},
]))


In [ ]:
# Baseline patrulha — reprodução explícita da lógica
calcular_risco(pts, str(XLSX))
risco = np.array([p["risco"] for p in pts])
total = float(risco.sum())
n_pat = int((risco >= LIMIAR).sum())  # 274 células canónicas

idx_sad = np.argsort(-risco)[:n_pat]
captura_sad = float(risco[idx_sad].sum() / total)

rng = np.random.default_rng(7)
sims = [float(risco[rng.choice(len(pts), n_pat, replace=False)].sum()/total) for _ in range(500)]
captura_rand = float(np.mean(sims))

ordenado = sorted(range(len(pts)), key=lambda i: pts[i]["dist_costa_km"])
step = max(1, len(ordenado)//n_pat)
idx_uni = ordenado[::step][:n_pat]
captura_uni = float(risco[idx_uni].sum() / total)

print(f"Células patrulha N={n_pat}")
print(f"Captura SAD: {100*captura_sad:.1f}% | Aleatório: {100*captura_rand:.1f}% | Uniforme: {100*captura_uni:.1f}%")
print(f"Ganho SAD/aleatório: {captura_sad/max(captura_rand,1e-6):.2f}×")

bl = baseline_patrulha(pts, n_pat)
fig_backtest(bt); fig_baseline(bl)
sens_lim = sensibilidade_limiar(pts)
fig_sensibilidade_limiar(sens_lim)

show_fig("21_backtest_temporal.png", "Fig. 21 — Backtest temporal")
show_fig("22_baseline_patrulha.png", "Fig. 22 — Baseline patrulha")
show_fig("25_sensibilidade_limiar.png", "Fig. 25 — Sensibilidade limiar")


In [ ]:
# Carregar validacao.json canónico (métricas da plataforma)
with open(OUTDIR/"validacao.json", encoding="utf-8") as f:
    val = json.load(f)

bl_json = val["baseline_patrulha"]
print("Métricas canónicas (plataforma):")
print(f"  n_celulas_patrulha = {bl_json['n_celulas_patrulha']}")
print(f"  ganho_sad_vs_aleatorio = {bl_json['ganho_sad_vs_aleatorio']}×")
print(f"  pct_risco_capturado = {bl_json['pct_risco_total_capturado_sad']}%")


## 6. Respostas SAD — Q1, Q2, Q3

In [ ]:
resp = val["resposta_objetivo"]
display(pd.DataFrame([
    ("Q1 — Onde patrulhar?", resp["Q1_onde"]["resposta"]),
    ("Q2 — Quantos AR5?", resp["Q2_quantos"]["resposta"]),
    ("Q3 — Onde colocar bases?", resp["Q3_bases"]["resposta"]),
], columns=["Pergunta", "Resposta"]))

print("\nZonas Q1:", resp["Q1_onde"].get("zonas_patrulha"))
print("Frota Q2:", resp["Q2_quantos"].get("frota_costeira"), "costeiros,",
      resp["Q2_quantos"].get("n_simultaneos"), "simultâneos")
print("Bases Q3 MCLP:", resp["Q3_bases"].get("bases"))


## 7. Ligação à plataforma web

A API (`plataforma/api`) importa `src/` e lê estes JSON em runtime:

| Ficheiro | Uso na plataforma |
|----------|-------------------|
| `resultados/validacao.json` | Painel Ciência, respostas SAD Q1–Q3 |
| `resultados/resultados.json` | Frota, MCLP, sensibilidade |
| `resultados/camadas_mapa.json` | Camadas do mapa (apreensões, IOM, clusters) |
| `resultados/ahp_pesos.json` | Sliders AHP no painel Ciência |
| `dados/processados/intensidades_reais.csv` | Grelha de risco no mapa |


In [ ]:
artefactos = [
    "validacao.json", "resultados.json", "camadas_mapa.json",
    "ahp_pesos.json", "demo_navios.json", "manifest.json",
]
for a in artefactos:
    p = OUTDIR / a
    if p.exists():
        with open(p, encoding="utf-8") as f:
            data = json.load(f)
        print(f"\n=== {a} ({p.stat().st_size//1024} KB) ===")
        if a == "validacao.json":
            print("  backtest ganho:", data["backtest_temporal"].get("ganho_relativo_limiar"))
            print("  baseline ganho:", data["baseline_patrulha"].get("ganho_sad_vs_aleatorio"))
        elif a == "resultados.json":
            b = data.get("cenarios",{}).get("B_alcance_AR5",{})
            print("  k recomendado:", b.get("k_recomendado"))
            print("  frota costeira:", b.get("frota_persistencia_costeira",{}).get("frota_total"))


## 8. Galeria — todas as figuras do relatório

In [ ]:
for p in sorted(FIGDIR.glob("*.png")):
    show_fig(p.name, p.stem.replace("_", " "))
if (DMFIG/"eda_panorama.png").exists():
    show_fig("eda_panorama.png", "EDA panorama", DMFIG)
if (FIGDIR/"24_ahp_pesos.png").exists():
    show_fig("24_ahp_pesos.png", "AHP pesos")


## Referência rápida — comandos equivalentes

```bash
# Pipeline completo em linha de comandos:
cd src
python -m dm.eda          # ou: python -c "from dm.eda import gerar; gerar()"
python -m dm.ahp_pesos
python main.py            # figuras 01–08 + resultados.json
python validacao.py       # figuras 21–22–25 + validacao.json

# Plataforma:
cd ../plataforma && ./start-mac.sh   # ou setup-win.ps1 no Windows
```
